# Pemrosesan Data Berita & PTA

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse

def scrape_kompas_article(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        r = requests.get(url, headers=headers)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error saat mengambil URL: {e}")
        return None

    soup = BeautifulSoup(r.content, "html.parser")

    # Judul
    judul = soup.select_one("h1.read__title")
    judul_text = judul.text.strip() if judul else "Tidak ditemukan judul"

    # Isi (100 kata)
    isi_elems = soup.select("div.read__content p")
    isi_text = " ".join([p.get_text(strip=True) for p in isi_elems]) if isi_elems else "Tidak ditemukan isi"
    words = isi_text.split()
    isi_100 = " ".join(words[:100]) + ("..." if len(words) > 100 else "")

    # Cari kategori
    kategori_text = "Tidak ditemukan kategori"

    kategori_meta = soup.find("meta", {"property": "article:section"})
    if kategori_meta and kategori_meta.get("content"):
        kategori_text = kategori_meta["content"].strip()

    if kategori_text == "Tidak ditemukan kategori":
        breadcrumb = soup.select("div.breadcrumb__link a")
        if breadcrumb and len(breadcrumb) > 1:
            kategori_text = breadcrumb[1].get_text(strip=True)

    if kategori_text == "Tidak ditemukan kategori":
        parsed = urlparse(url)
        subdomain = parsed.netloc.split(".")[0]
        if subdomain and subdomain != "www" and subdomain != "kompas":
            kategori_text = subdomain.capitalize()

    return {
        "Judul": judul_text,
        "Isi (100 kata)": isi_100,
        "Kategori": kategori_text
    }

# Daftar URL
urls_to_test = [
    "https://lifestyle.kompas.com/read/2025/08/31/100000720/waspadai-kelelahan-mental-akibat-kebanyakan-berita-negatif-",
    "https://money.kompas.com/read/2025/08/18/134653726/pendidikan-kewirausahaan-yang-merdeka",
    "https://health.kompas.com/read/25H19130000068/dokter--olahraga-bisa-turunkan-risiko-kanker-asal-rutin-dan-benar",
    "https://nasional.kompas.com/read/2025/09/03/18022971/peristiwa-gas-air-mata-unisba-mendikti-janjikan-pendampingan-dan",
    "https://nasional.kompas.com/read/2025/08/19/07251961/harapan-dan-catatan-soal-anggaran-pendidikan-terbesar-sepanjang-sejarah-ri",
    "https://travel.kompas.com/read/2025/09/04/210100827/berdarah-belanda-depok-pesepak-bola-miliano-jonathans-resmi-jadi-wni"
]

# Scraping data dari URL
results = [scrape_kompas_article(u) for u in urls_to_test if scrape_kompas_article(u)]
df = pd.DataFrame(results)

# Simpan DataFrame ke file CSV
df.to_csv("kompas_articles.csv", index=False)

from IPython.display import display
print("Data berhasil disimpan ke file 'kompas_articles.csv'")
display(df)

Data berhasil disimpan ke file 'kompas_articles.csv'


,Judul,Isi (100 kata),Kategori
0,Waspadai Kelelahan Mental akibat Kebanyakan Be...,KOMPAS.com -Berbagai informasi peristiwa terba...,Lifestyle
1,Pendidikan Kewirausahaan yang Merdeka,"SETIAP17 Agustus, Masyarakat Indonesia merayak...",Money
2,"Dokter: Olahraga Bisa Turunkan Risiko Kanker, ...",KOMPAS.com –Health Management Specialist Corpo...,Health
3,"Peristiwa Gas Air Mata Unisba, Mendikti Janjik...","JAKARTA, KOMPAS.com- Menteri Pendidikan Tinggi...",Nasional
4,Harapan dan Catatan soal Anggaran Pendidikan T...,"JAKARTA, KOMPAS.com- Presiden Prabowo Subianto...",Nasional
5,"Berdarah Belanda Depok, Pesepak Bola Miliano J...",KOMPAS.com -Kementerian Hukum Republik Indones...,Travel


In [3]:
!pip install pandas
!pip install nltk
!pip install Sastrawi
!pip install Sastrawi
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
import pandas as pd
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# --- MEMBACA BERITA PERTAMA DARI FILE CSV ---
df = pd.read_csv('kompas_articles.csv')
teks = df.loc[0, 'Isi (100 kata)']

print("--- Berita Asli dari CSV ---")
print(teks)
print("-" * 30)

# 1. Penghapusan kata umum (stopword)
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
teks_no_stop = stopword_remover.remove(teks)
print("1. Penghapusan kata umum (stopword):")
print(teks_no_stop)
print("-" * 30)

# 2. Menghilangkan simbol tanda baca (punctuation)
teks_no_punct = teks_no_stop.translate(str.maketrans('', '', string.punctuation))
print("2. Menghilangkan simbol tanda baca:")
print(teks_no_punct)
print("-" * 30)

# 3. Cek ejaan pembakuan kata
# Kamus pembakuan kata (contoh sederhana)
kata_baku = {
    'diumumin': 'diumumkan',
    'resmi': 'resmi',
    'enggak': 'tidak',
    'syuting': 'syuting',
    'berlarii': 'lari',
    'sperti': 'seperti'
}

def standardisasi_kata(teks):
    teks_baru = []
    for kata in teks.split():
        teks_baru.append(kata_baku.get(kata, kata))
    return ' '.join(teks_baru)

teks_baku = standardisasi_kata(teks_no_punct.lower())
print("3. Cek ejaan pembakuan kata:")
print(teks_baku)
print("-" * 30)

# 4. Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()
teks_stemmed = stemmer.stem(teks_baku)
print("4. Stemming dan lematisasi:")
print(teks_stemmed)
print("-" * 30)

# 5. Tokenisasi
tokens = word_tokenize(teks_stemmed)
print("5. Tokenisasi:")
print(tokens)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


--- Berita Asli dari CSV ---
KOMPAS.com -Berbagai informasi peristiwa terbaru sekarang bisa kita dapatkan tanpa henti, baik melalui media arus utama atau media sosial. Tanpa disadari kondisi itu berpengaruh besar pada kondisi mental seseorang. Para ahli menyebut fenomena ini sebagai "headline stress disorder" untuk menggambarkan kondisi stres akibat terlalu sering terpapar berita bernuansa negatif. Psikolog sosial Dicky Pelupessy Ph.D menyebutkan, emosi sedih, gelisah, atau marah yang timbul tersebut sebenarnya hal yang normal. "Ini adalah perasaan yang normal, karena peristiwa politik, apalagi yang luar biasa, tidak terjadi setiap hari, sehingga akan membangkitkan emosi negatif," katanya ketika dihubungi Kompas.com (29/8/2025). Baca juga:Mengapa Bisa Cemas Setelah Lihat Berita...
------------------------------
1. Penghapusan kata umum (stopword):
KOMPAS.com -Berbagai informasi peristiwa terbaru sekarang kita dapatkan henti, baik melalui media arus utama media sosial. Tanpa disadari ko

# **PEMROSESAN DATA PTA**

In [11]:
import pandas as pd
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# --- MEMBACA DATA DARI FILE CSV ---
df = pd.read_csv('crawlingPTA.csv', engine='python', on_bad_lines='skip')

# Ambil hanya kolom abstrak bahasa Indonesia dari seluruh data
abstrak_list = df['abstrak_bindonesia'].dropna().tolist()

print(f"Jumlah data abstrak Bahasa Indonesia: {len(abstrak_list)}")
print("=" * 50)

# --- PROSES SATU PERSATU ---
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Kamus pembakuan kata (contoh)
kata_baku = {
    'diumumin': 'diumumkan',
    'resmi': 'resmi',
    'enggak': 'tidak',
    'syuting': 'syuting',
    'berlarii': 'lari',
    'sperti': 'seperti'
}

def standardisasi_kata(teks):
    teks_baru = []
    for kata in teks.split():
        teks_baru.append(kata_baku.get(kata, kata))
    return ' '.join(teks_baru)

# Looping semua abstrak
for i, teks in enumerate(abstrak_list[:5], start=1):  # tampilkan contoh 5 dulu
    print(f"\n=== Abstrak {i} (Asli) ===")
    print(teks)

    # 1. Stopword removal
    teks_no_stop = stopword_remover.remove(teks)
    print("\n1. Stopword Removal:")
    print(teks_no_stop)

    # 2. Hapus tanda baca
    teks_no_punct = teks_no_stop.translate(str.maketrans('', '', string.punctuation))
    print("\n2. Hapus Tanda Baca:")
    print(teks_no_punct)

    # 3. Standardisasi kata
    teks_baku = standardisasi_kata(teks_no_punct.lower())
    print("\n3. Standardisasi Kata:")
    print(teks_baku)

    # 4. Stemming
    teks_stemmed = stemmer.stem(teks_baku)
    print("\n4. Stemming:")
    print(teks_stemmed)

    # 5. Tokenisasi
    tokens = word_tokenize(teks_stemmed)
    print("\n5. Tokenisasi:")
    print(tokens)
    print("=" * 50)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Jumlah data abstrak Bahasa Indonesia: 893

=== Abstrak 1 (Asli) ===
ABSTRAK

       Implementasi Fungsi Legislasi DPRD Kabupaten Bangkalan Periode 2009-2014 Dalam Pembentukan Peraturan Daerah menurut ketentuan Undang-Undang Dasar Negara Republik Indonesia Tahun 1945 maupun Undang-Undang Republik Indonesia Nomor 12 Tahun 2011 Tentang Pembentukan Peraturan Perundang-undangan dan Undang-Undang Republik Indonesia Nomor 32 Tahun 2004 Tentang Pemerintahan Daerah menentukan bahwa fungsi legislasi berada ditangan DPRD. Fungsi Legislasi yang dimiliki oleh DPRD merupakan fungsi pembentukan peraturan daerah (perda). Begitupula dengan DPRD Kabupaten Bangkalan yang memiliki fungsi legislasi tersebut dalam pembuatan peraturan daerah. 
       Metode penelitian yang digunakan adalah yuridis normatif. Pendekatan masalah menggunakan pendekatan perundang-undangan (statute approach). 
       Hasil penelitian ini menunjukkan bahwa dalam pelaksanaan fungsi legislasi DPRD Kabupaten Bangkalan dalam pembentuka